# Multi-scenario portfolio stress comparison

Canonical hazard rows already carry `pathway` and `horizon`. Pack several
sensitivity cases into one Parquet file and the same
`HazardDataset.for_assets(...).return_periods(...).write_parquet(...)` call emits
one row per asset × scenario — no separate evaluation code path.

This offline notebook derives two transparent tail stresses from the checked-in
historical WRI fixture:

- **historical / 1980** — fitted baseline
- **tail_stress_10pct / 2050** — fitted tail scale × 1.10
- **tail_stress_20pct / 2050** — fitted tail scale × 1.20

These are sensitivity cases for demonstrating scenario mechanics, not calibrated
climate-model projections.


In [ ]:
from hashlib import sha256
from pathlib import Path

import pandas as pd
import plotly.graph_objects as go
import plotly.io as pio
import pyarrow as pa
import pyarrow.parquet as pq

from crc_sdk.connectors import (
    read_hazard_dataset,
    read_hazard_metadata,
    write_hazard_dataset,
)
from crc_sdk.workflows import ExecutionOptions, HazardDataset, return_period_value_columns

pio.renderers.default = "jupyterlab+png"

DATA_DIR = Path("data")
DATA_DIR.mkdir(exist_ok=True)
FIXTURE_DIR = Path("../fixtures/os_climate")

HAZARD_NAME = "RiverineInundation"
RETURN_PERIODS = [100, 500, 1000]
SCENARIOS = [
    ("historical", 1980, 1.0),
    ("tail_stress_10pct", 2050, 1.1),
    ("tail_stress_20pct", 2050, 1.2),
]

BASELINE_PATH = FIXTURE_DIR / "hazard.parquet"
ASSETS_PATH = FIXTURE_DIR / "assets.parquet"
HAZARD_PATH = DATA_DIR / "wri_riverine_stress_scenarios_cologne.parquet"
EVAL_PATH = DATA_DIR / "cologne_portfolio_stress_scenarios.parquet"


## 1. Assets


In [ ]:
assets = pq.read_table(ASSETS_PATH)
print(f"{assets.num_rows} assets <- {ASSETS_PATH}")
assets.to_pandas()


## 2. Build several local stress scenarios in one hazard file

Copy the baseline rows, tag each case with its `pathway` and `horizon`, and scale
the continuous tail. `write_hazard_dataset` validates the resulting canonical
rows; the row key includes scenario dimensions, so all cases coexist cleanly.


In [ ]:
baseline = read_hazard_dataset(BASELINE_PATH)
metadata = read_hazard_metadata(BASELINE_PATH)
rows = []
for pathway, horizon, scale_multiplier in SCENARIOS:
    for source in baseline.to_pylist():
        row = dict(source)
        row["pathway"] = pathway
        row["horizon"] = horizon
        row["curve_scale"] = float(row["curve_scale"]) * scale_multiplier
        row["source_id"] = sha256(
            f"{source['source_id']}:{pathway}:{horizon}".encode()
        ).hexdigest()
        rows.append(row)

stress_table = pa.Table.from_pylist(rows, schema=baseline.schema)
write_hazard_dataset(stress_table, HAZARD_PATH, metadata)
hazard = read_hazard_dataset(HAZARD_PATH)
(
    hazard.to_pandas()
    .groupby(["pathway", "horizon"], as_index=False)
    .size()
    .rename(columns={"size": "rows"})
)


## 3. Evaluate once across every scenario

Omit pathway/horizon filters so the join emits one row per asset × scenario.
Use `.select(pathways=[...], horizons=[...])` when you want a single slice instead.


In [ ]:
result = (
    HazardDataset.local(HAZARD_PATH)
    .for_assets(ASSETS_PATH)
    .select(hazard_names=[HAZARD_NAME])
    .return_periods(RETURN_PERIODS)
    .write_parquet(EVAL_PATH, execution=ExecutionOptions(max_workers=1))
)
evaluated = pq.read_table(EVAL_PATH).to_pandas()
print(f"evaluated rows: {result.row_count} (assets × scenarios)")
evaluated[
    ["asset_id", "pathway", "horizon", *result.value_columns]
].sort_values(["horizon", "pathway", "asset_id"])


## 4. Visualize


In [ ]:
design_col = return_period_value_columns([100])[0]
evaluated = evaluated.assign(
    scenario_label=evaluated["pathway"] + "/" + evaluated["horizon"].astype(str)
)

fig = go.Figure()
for asset_id, frame in evaluated.groupby("asset_id"):
    frame = frame.sort_values(["horizon", "pathway"])
    fig.add_trace(
        go.Bar(
            name=asset_id,
            x=frame["scenario_label"],
            y=frame[design_col],
        )
    )
fig.update_layout(
    barmode="group",
    title="100-year flood depth by local tail-stress scenario",
    xaxis_title="Pathway / horizon",
    yaxis_title="Depth (m)",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig.show()


In [ ]:
portfolio = (
    evaluated.groupby(["pathway", "horizon", "scenario_label"], as_index=False)
    .agg(mean_depth_m=(design_col, "mean"), total_depth_m=(design_col, "sum"))
    .sort_values(["horizon", "pathway"])
)
fig_port = go.Figure(
    go.Bar(
        x=portfolio["scenario_label"],
        y=portfolio["mean_depth_m"],
        marker_color=["#2171B5", "#6BAED6", "#D94801"],
        text=[f"{v:.3f} m" for v in portfolio["mean_depth_m"]],
        textposition="outside",
    )
)
fig_port.update_layout(
    title="Portfolio-mean 100-year depth by scenario",
    xaxis_title="Pathway / horizon",
    yaxis_title="Mean depth (m)",
    margin=dict(l=60, r=40, t=40, b=60),
)
fig_port.show()
portfolio


## Scaling this up

[`pipelines/multi_scenario_pipeline.py`](../pipelines/multi_scenario_pipeline.py)
is the headless twin. Add local stress definitions — or replace them with real
canonical model scenarios later — without changing the portfolio evaluation
call. The distinction should remain explicit: these scale stresses demonstrate
mechanics and are not calibrated projections.
